# Agent & System Interoperability

`AnalysisTree` can be serialized to and from JSON, making it straightforward to:

- **Persist** analysis plans alongside results
- **Exchange** plans between services or processes
- **Generate or modify** analysis plans with an AI agent

This notebook walks through the four key methods: `to_dict`, `to_json`, `from_dict`, and `from_json`.

In [ ]:
import numpy as np
import pandas as pd
from pyMyriad import AnalysisTree

# Make string expressions work without passing environ explicitly
AnalysisTree.set_default_environ({"np": np, "pd": pd})

In [ ]:
# Sample dataset used throughout this notebook
np.random.seed(42)
df = pd.DataFrame({
    "Gender":  np.random.choice(["M", "F"], 200),
    "Country": np.random.choice(["US", "UK", "Canada"], 200),
    "Age":     np.random.randint(18, 70, 200),
    "Income":  np.random.normal(50_000, 15_000, 200),
})
df.head()

## 1 — Serialize a tree to JSON

Build any `AnalysisTree` — including ones with lambda functions — then call `to_json()`.
Lambda bodies are automatically extracted as plain string expressions so the output is fully human-readable.

In [ ]:
tree = (AnalysisTree()
    .split_by("df.Gender", label="Gender")
    .split_by("df.Country", label="Country")
    .analyze_by(
        n=lambda df: len(df),
        mean_income=lambda df: np.mean(df.Income),
        std_income=lambda df: np.std(df.Income),
        label="stats",
    ))

print(tree)  # human-readable tree structure

In [ ]:
json_str = tree.to_json()
print(json_str)

Notice that the lambdas (e.g. `lambda df: np.mean(df.Income)`) have been reduced to their body expression (`"np.mean(df.Income)"`).
This makes the JSON fully portable — no Python closures, no pickle.

## 2 — Deserialize and run

`from_json()` accepts either a raw JSON string or a file path — it auto-detects which one you pass.

In [ ]:
loaded_tree = AnalysisTree.from_json(json_str)
print(loaded_tree)  # identical structure to the original

In [ ]:
# Run the reconstructed tree — results are numerically identical to the original

from pyMyriad import simple_table

result = loaded_tree.run(df)

simple_table(result)

## 3 — File I/O

Pass a file path to `to_json()` to persist the plan.  Load it back later with `from_json()`.

In [ ]:
tree.to_json("analysis_plan.json")

# Later — or in a different process / service:
restored = AnalysisTree.from_json("analysis_plan.json")
print(restored)

## 4 — Agent workflow: generate → modify → run

A common pattern when working with AI agents:

1. The agent receives a description of the desired analysis and produces a JSON plan.
2. You (or another service) validate and optionally tweak the JSON.
3. pyMyriad loads the plan and runs it against the actual data.

The cell below simulates step 1 — an agent generating a plan from scratch as a Python `dict`.

In [ ]:
# --- Simulated agent output ---
# An agent produced this JSON plan based on the request:
# "Stratify by Gender, then compare each country's mean income against the US baseline"
agent_plan = {
    "type": "AnalysisTree",
    "denom": None,
    "nodes": [
        {
            "type": "SplitNode",
            "label": "Gender",
            "drop_empty": False,
            "expr": "df.Gender",
            "nodes": [
                {
                    "type": "SplitNode",
                    "label": "Country",
                    "drop_empty": False,
                    "expr": "df.Country",
                    "nodes": [
                        {
                            "type": "AnalysisNode",
                            "label": "summary",
                            "termination": True,
                            "analysis": {
                                "n": "len(df)",
                                "mean_income": "np.mean(df.Income)",
                            }
                        },
                        {
                            "type": "CrossAnalysisNode",
                            "label": "vs_US",
                            "termination": True,
                            "ref_lvl": "US",
                            "analysis": {
                                "delta": "np.mean(df.Income) - np.mean(ref_df.Income)"
                            }
                        }
                    ]
                }
            ]
        }
    ]
}

agent_tree = AnalysisTree.from_dict(agent_plan)
print(agent_tree)

In [ ]:
agent_result = agent_tree.run(df)
simple_table(agent_result)

## 5 — Round-trip correctness check

Verify that serializing and deserializing a tree produces numerically identical results.

In [ ]:
original_result = tree.run(df)
roundtrip_result = AnalysisTree.from_json(tree.to_json()).run(df)

# Compare a few values
for gender in ["M", "F"]:
    for country in ["US", "UK", "Canada"]:
        try:
            orig = original_result["Gender"][gender]["Country"][country]["stats"].summary
            rt   = roundtrip_result["Gender"][gender]["Country"][country]["stats"].summary
            match = np.isclose(orig["mean_income"], rt["mean_income"]) and orig["n"] == rt["n"]
            print(f"{gender}/{country}: match={match}  n={orig['n']}  mean_income={orig['mean_income']:.0f}")
        except KeyError:
            pass  # empty group for this combination

## Summary

| Method | Direction | Accepts |
|---|---|---|
| `tree.to_dict()` | Python → dict | — |
| `tree.to_json(path=None)` | Python → JSON string / file | optional file path |
| `AnalysisTree.from_dict(d)` | dict → Python | plain dict |
| `AnalysisTree.from_json(source)` | JSON → Python | string **or** file path |

**Lambda note:** lambdas are serialized as their body expression string (e.g. `"np.mean(df.Income)"`).
After deserialization, expressions are strings — ensure `np`, `pd`, etc. are available via `environ` or `set_default_environ()`.